# Silver -> Azure SQL (OLTP source seed)

Reverse ETL that populates the **Azure SQL** transactional schema
(`deploy/azuresql/schema`) from the `retail_lakehouse` **silver** Delta tables.
The Azure SQL database is intended to be **mirrored back into the lakehouse
`bronze` schema**, so this notebook seeds that operational source.

## Data flow
```
retail_lakehouse.silver.*  (Delta)  ->  Azure SQL  retail.*  (OLTP)
```

## What it does
- Reads each silver fact/dim table.
- Reshapes to the denormalized OLTP model (renames, cents -> DECIMAL dollars,
  unions store+DC inventory, folds promotions onto sales/lines).
- Bulk-writes to Azure SQL via the Spark JDBC / MSSQL Spark connector in
  **foreign-key-safe order**.

## Prerequisites
- The OLTP schema is already deployed (`deploy/azuresql/deploy-schema.ps1`).
- Fabric workspace has network line-of-sight to the Azure SQL server and the
  identity/credential below can write to the `retail` schema.

## Column convention
`snake_case` throughout (matches the silver contract and CLAUDE.md).

In [ ]:
from pyspark.sql import functions as F
from datetime import datetime, timezone
import os

try:
    import notebookutils  # Fabric
except ImportError:  # pragma: no cover - older runtimes
    import mssparkutils as notebookutils

In [ ]:
# =============================================================================
# PARAMETERS  (Fabric "parameters" cell: override via notebook/job parameters
# or environment variables. Composed/derived values live in the next cell so
# they pick up any injected parameter overrides.)
# =============================================================================

def get_env(var_name, default=None):
    return os.environ.get(var_name, default)

# --- source lakehouse ---
LAKEHOUSE_NAME = get_env("LAKEHOUSE_NAME", "retail_lakehouse")
SILVER_DB      = get_env("SILVER_DB", "silver")

# --- target Azure SQL ---
AZURE_SQL_SERVER   = get_env("AZURE_SQL_SERVER", "")     # e.g. myserver.database.windows.net
AZURE_SQL_DATABASE = get_env("AZURE_SQL_DATABASE", "")   # e.g. retaildb
TARGET_SCHEMA      = get_env("TARGET_SCHEMA", "retail")

# --- authentication ---
# AUTH_MODE = "aad_token" (workspace/user identity) or "sql". An AAD token
# expires (~60-75 min); long writes are chunked with a fresh token per chunk
# (see MAX_ROWS_PER_WRITE). "sql" auth has no token to expire.
AUTH_MODE       = get_env("AUTH_MODE", "aad_token")
AAD_AUDIENCE    = get_env("AAD_AUDIENCE", "https://database.windows.net/.default")
SQL_USER        = get_env("SQL_USER", "")
SQL_PASSWORD    = get_env("SQL_PASSWORD", "")          # avoid; prefer Key Vault
KEY_VAULT_URL   = get_env("KEY_VAULT_URL", "")         # https://<kv>.vault.azure.net/
SQL_SECRET_NAME = get_env("SQL_SECRET_NAME", "")       # secret holding the SQL password

# --- write behaviour ---
# WRITE_FORMAT "jdbc" (default) does column-list INSERTs, so the server fills the
# IDENTITY PKs and loaded_at default the transforms omit. The BULK COPY connector
# ("com.microsoft.sqlserver.jdbc.spark") needs the full target column set and is
# NOT compatible with those columns as-is -- see the README before switching.
WRITE_FORMAT         = get_env("WRITE_FORMAT", "jdbc")
BATCH_SIZE           = int(get_env("BATCH_SIZE", "100000"))
# Bound each physical write so it finishes inside the AAD token lifetime AND
# commits before a small Azure SQL tier drops the connection under load. Large
# tables split into deterministic chunks, each written with a fresh token; keep
# this modest (1M) so per-chunk commits stay small on constrained tiers.
MAX_ROWS_PER_WRITE   = int(get_env("MAX_ROWS_PER_WRITE", "1000000"))
# Cap concurrent JDBC connections into the single Azure SQL target. Too many
# parallel bulk inserts overwhelm a small tier (LATCH_SH contention, dropped
# connections); 8 is a safe default. Raise on larger vCore tiers.
WRITE_PARTITIONS     = int(get_env("WRITE_PARTITIONS", "8"))
TRUNCATE_BEFORE_LOAD = str(get_env("TRUNCATE_BEFORE_LOAD", "true")).lower() == "true"
ONLY_TABLES          = get_env("ONLY_TABLES", "")     # comma-separated string; parsed next cell
LOADED_AT            = get_env("LOADED_AT", "")        # blank -> set to run time in the next cell


In [ ]:
# =============================================================================
# DERIVED CONFIG  (runs after any injected parameters)
# =============================================================================

assert AZURE_SQL_SERVER and AZURE_SQL_DATABASE, "Set AZURE_SQL_SERVER and AZURE_SQL_DATABASE"

# comma-separated string -> list of OLTP tables (default = all)
if isinstance(ONLY_TABLES, str):
    ONLY_TABLES = [t.strip() for t in ONLY_TABLES.split(",") if t.strip()]

# one UTC batch timestamp per run, stamped on every row (unless supplied)
if not LOADED_AT:
    LOADED_AT = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]

JDBC_URL = (
    f"jdbc:sqlserver://{AZURE_SQL_SERVER};databaseName={AZURE_SQL_DATABASE};"
    "encrypt=true;trustServerCertificate=false;loginTimeout=60;"
    "connectRetryCount=3;connectRetryInterval=10;"
)
print(f"source={LAKEHOUSE_NAME}.{SILVER_DB} -> target={AZURE_SQL_SERVER}/{AZURE_SQL_DATABASE}.{TARGET_SCHEMA}")
print(f"auth={AUTH_MODE}  write_format={WRITE_FORMAT}  max_rows_per_write={MAX_ROWS_PER_WRITE}  truncate_before_load={TRUNCATE_BEFORE_LOAD}")
print(f"batch loaded_at={LOADED_AT}")


In [ ]:
# =============================================================================
# CONNECTION / AUTH
# =============================================================================

def get_auth_options():
    """Return the JDBC auth options for the configured AUTH_MODE."""
    if AUTH_MODE == "aad_token":
        token = notebookutils.credentials.getToken(AAD_AUDIENCE)
        return {"accessToken": token}
    password = SQL_PASSWORD
    if not password and KEY_VAULT_URL and SQL_SECRET_NAME:
        password = notebookutils.credentials.getSecret(KEY_VAULT_URL, SQL_SECRET_NAME)
    assert SQL_USER and password, "SQL auth requires SQL_USER and a password (SQL_PASSWORD or Key Vault)"
    return {"user": SQL_USER, "password": password}


import time


def _retry(fn, what, attempts=5, base_delay=5):
    """Run fn() with exponential backoff. Use only for idempotent operations
    (opening a connection, or a write we can safely re-run after clearing)."""
    for attempt in range(1, attempts + 1):
        try:
            return fn()
        except Exception as exc:  # noqa: BLE001 - Py4J/JDBC errors are opaque
            if attempt == attempts:
                raise
            delay = base_delay * (2 ** (attempt - 1))
            print(f"     {what} failed ({attempt}/{attempts}): {exc}; retry in {delay}s")
            time.sleep(delay)


def _jdbc_connection():
    """Open a raw JDBC connection through the JVM for DDL/DML (reset, FK toggles).
    The open is retried with backoff so a transient drop does not abort a reset."""
    jvm = spark._sc._gateway.jvm  # noqa: F821 - spark provided by Fabric
    jvm.java.lang.Class.forName("com.microsoft.sqlserver.jdbc.SQLServerDriver")

    def _open():
        props = jvm.java.util.Properties()
        for key, value in get_auth_options().items():  # fresh token on each try
            props.setProperty(key, str(value))
        return jvm.java.sql.DriverManager.getConnection(JDBC_URL, props)

    return _retry(_open, "jdbc connect")


def execute_sql(statements):
    """Execute a list of T-SQL statements on one connection."""
    conn = _jdbc_connection()
    try:
        stmt = conn.createStatement()
        for sql in statements:
            stmt.execute(sql)
        stmt.close()
    finally:
        conn.close()

In [ ]:
# =============================================================================
# TRANSFORM HELPERS
# =============================================================================

def silver(table):
    return spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.{table}")  # noqa: F821


def money_cents(col):
    """Integer cents -> DECIMAL(19,4) dollars."""
    return (F.col(col) / F.lit(100.0)).cast("decimal(19,4)")

In [ ]:
# =============================================================================
# MASTER / REFERENCE TRANSFORMS
# =============================================================================

def build_geographies():
    return silver("dim_geographies").select(
        F.col("ID").alias("geography_id"),
        F.col("City").alias("city"),
        F.col("State").alias("state"),
        F.col("ZipCode").alias("zip_code"),
        F.col("District").alias("district"),
        F.col("Region").alias("region"),
    )


def build_customers():
    return silver("dim_customers").select(
        F.col("ID").alias("customer_id"),
        F.col("FirstName").alias("first_name"),
        F.col("LastName").alias("last_name"),
        F.col("Address").alias("address"),
        F.col("GeographyID").alias("geography_id"),
        F.col("LoyaltyCard").alias("loyalty_card"),
        F.col("Phone").alias("phone"),
        F.col("BLEId").alias("ble_id"),
        F.col("AdId").alias("ad_id"),
    )


def build_stores():
    return silver("dim_stores").select(
        F.col("ID").alias("store_id"),
        F.col("StoreNumber").alias("store_number"),
        F.col("Address").alias("address"),
        F.col("GeographyID").alias("geography_id"),
        F.col("tax_rate").cast("decimal(6,4)").alias("tax_rate"),
        F.col("volume_class"),
        F.col("store_format"),
        F.col("operating_hours"),
        F.col("daily_traffic_multiplier").cast("decimal(9,4)").alias("daily_traffic_multiplier"),
    )


def build_distribution_centers():
    return silver("dim_distribution_centers").select(
        F.col("ID").alias("dc_id"),
        F.col("DCNumber").alias("dc_number"),
        F.col("Address").alias("address"),
        F.col("GeographyID").alias("geography_id"),
    )


def build_trucks():
    return silver("dim_trucks").select(
        F.col("ID").alias("truck_id"),
        F.col("LicensePlate").alias("license_plate"),
        F.col("Refrigeration").alias("refrigeration"),
        F.col("DCID").cast("long").alias("dc_id"),
    )


def build_products():
    return silver("dim_products").select(
        F.col("ID").alias("product_id"),
        F.col("ProductName").alias("product_name"),
        F.col("Brand").alias("brand"),
        F.col("Company").alias("company"),
        F.col("Department").alias("department"),
        F.col("Category").alias("category"),
        F.col("Subcategory").alias("subcategory"),
        F.col("Cost").cast("decimal(19,4)").alias("cost"),
        F.col("MSRP").cast("decimal(19,4)").alias("msrp"),
        F.col("SalePrice").cast("decimal(19,4)").alias("sale_price"),
        F.col("RequiresRefrigeration").alias("requires_refrigeration"),
        F.col("LaunchDate").alias("launch_date"),
        F.col("taxability"),
        F.col("Tags").alias("tags"),
    )

In [ ]:
# =============================================================================
# COMMERCE TRANSFORMS
# =============================================================================

def build_sales():
    receipts = silver("fact_receipts")
    # receipt-level promo folded onto the header (one code per receipt)
    promo = (
        silver("fact_promotions")
        .groupBy("receipt_id_ext")
        .agg(F.first("promo_code", ignorenulls=True).alias("promo_code"))
    )
    return (
        receipts.join(promo, "receipt_id_ext", "left").select(
            F.col("receipt_id_ext").alias("receipt_id"),
            F.col("store_id"),
            F.col("customer_id"),
            F.col("event_ts").alias("sale_ts"),
            F.col("receipt_type"),
            F.col("tender_type"),
            F.col("payment_method"),
            money_cents("subtotal_cents").alias("subtotal_amount"),
            F.col("discount_amount").cast("decimal(19,4)").alias("discount_amount"),
            money_cents("tax_cents").alias("tax_amount"),
            money_cents("total_cents").alias("total_amount"),
            F.col("promo_code"),
            F.col("trace_id"),
        )
    )


def build_sale_lines():
    lines = silver("fact_receipt_lines")
    # line-level promo discount folded onto the line
    promo = (
        silver("fact_promo_lines")
        .select(
            F.col("receipt_id_ext").alias("pl_receipt"),
            F.col("line_number").alias("pl_line"),
            (F.col("discount_cents") / F.lit(100.0)).cast("decimal(19,4)").alias("discount_amount"),
        )
        .groupBy("pl_receipt", "pl_line")
        .agg(F.sum("discount_amount").alias("discount_amount"))
    )
    join_cond = (lines["receipt_id_ext"] == promo["pl_receipt"]) & (lines["line_num"] == promo["pl_line"])
    return (
        lines.join(promo, join_cond, "left").select(
            lines["receipt_id_ext"].alias("receipt_id"),
            lines["line_num"].alias("line_number"),
            lines["product_id"],
            lines["quantity"].cast("int").alias("quantity"),
            money_cents("unit_cents").alias("unit_price"),
            money_cents("ext_cents").alias("extended_price"),
            lines["promo_code"],
            promo["discount_amount"],
        )
    )


def build_online_orders():
    return silver("fact_online_order_headers").select(
        F.col("order_id_ext").alias("order_id"),
        F.col("customer_id"),
        F.col("event_ts").alias("order_ts"),
        money_cents("subtotal_cents").alias("subtotal_amount"),
        money_cents("tax_cents").alias("tax_amount"),
        money_cents("total_cents").alias("total_amount"),
        F.col("payment_method"),
    )


def build_online_order_lines():
    return silver("fact_online_order_lines").select(
        F.col("order_id"),
        F.col("line_num").alias("line_number"),
        F.col("product_id"),
        F.col("quantity").cast("int").alias("quantity"),
        money_cents("unit_cents").alias("unit_price"),
        money_cents("ext_cents").alias("extended_price"),
        F.col("promo_code"),
        F.col("fulfillment_mode"),
        F.col("fulfillment_status"),
        F.col("node_type"),
        F.col("node_id"),
        F.col("picked_ts"),
        F.col("shipped_ts"),
        F.col("delivered_ts"),
    )


def build_payments():
    return silver("fact_payments").select(
        F.col("receipt_id_ext").alias("receipt_id"),
        F.col("order_id_ext").alias("order_id"),
        F.col("event_ts").alias("payment_ts"),
        F.col("payment_method"),
        money_cents("amount_cents").alias("amount"),
        F.col("transaction_id"),
        F.col("status"),
        F.col("decline_reason"),
        F.col("processing_time_ms"),
        F.col("store_id"),
        F.col("customer_id"),
    )

In [ ]:
# =============================================================================
# SUPPLY-CHAIN TRANSFORMS
# =============================================================================

def build_inventory_transactions():
    store = silver("fact_store_inventory_txn").select(
        F.lit("STORE").alias("location_type"),
        F.col("store_id").cast("long").alias("store_id"),
        F.lit(None).cast("long").alias("dc_id"),
        F.col("product_id"),
        F.col("event_ts").alias("txn_ts"),
        F.col("txn_type"),
        F.col("quantity").cast("int").alias("quantity"),
        F.col("balance").cast("int").alias("balance"),
        F.col("source"),
        F.col("trace_id"),
    )
    dc = silver("fact_dc_inventory_txn").select(
        F.lit("DC").alias("location_type"),
        F.lit(None).cast("long").alias("store_id"),
        F.col("dc_id").cast("long").alias("dc_id"),
        F.col("product_id"),
        F.col("event_ts").alias("txn_ts"),
        F.col("txn_type"),
        F.col("quantity").cast("int").alias("quantity"),
        F.col("balance").cast("int").alias("balance"),
        F.col("Source").alias("source"),
        F.col("trace_id"),
    )
    return store.unionByName(dc)


def build_reorders():
    return silver("fact_reorders").select(
        F.col("event_ts").alias("reorder_ts"),
        F.col("store_id").cast("long").alias("store_id"),
        F.col("dc_id").cast("long").alias("dc_id"),
        F.col("product_id"),
        F.col("current_quantity").cast("int").alias("current_quantity"),
        F.col("reorder_quantity").cast("int").alias("reorder_quantity"),
        F.col("reorder_point").cast("int").alias("reorder_point"),
        F.col("priority"),
        F.col("trace_id"),
    )


def build_stockouts():
    return silver("fact_stockouts").select(
        F.col("event_ts").alias("stockout_ts"),
        F.col("StoreID").cast("long").alias("store_id"),
        F.col("DCID").cast("long").alias("dc_id"),
        F.col("ProductID").cast("long").alias("product_id"),
        F.col("LastKnownQuantity").cast("int").alias("last_known_quantity"),
        F.col("trace_id"),
    )


def build_shipment_movements():
    return silver("fact_truck_moves").select(
        F.col("shipment_id").alias("shipment_number"),
        F.col("truck_id").cast("long").alias("truck_id"),
        F.col("dc_id").cast("long").alias("dc_id"),
        F.col("store_id").cast("long").alias("store_id"),
        F.col("status"),
        F.col("event_ts"),
        F.col("eta"),
        F.col("etd"),
        F.col("departure_time"),
        F.col("actual_unload_duration").cast("decimal(9,2)").alias("actual_unload_duration"),
        F.col("trace_id"),
    )


def build_shipment_lines():
    return silver("fact_truck_inventory").select(
        F.col("shipment_id").alias("shipment_number"),
        F.col("truck_id").cast("long").alias("truck_id"),
        F.col("product_id"),
        F.col("quantity").cast("int").alias("quantity"),
        F.col("action"),
        F.col("location_id").cast("long").alias("location_id"),
        F.col("location_type"),
        F.col("event_ts"),
        F.col("trace_id"),
    )

In [ ]:
# =============================================================================
# LOAD PLAN  (parent -> child; FK-safe order)
# =============================================================================

LOAD_PLAN = [
    ("geographies",           build_geographies),
    ("products",              build_products),
    ("customers",             build_customers),
    ("stores",                build_stores),
    ("distribution_centers",  build_distribution_centers),
    ("trucks",                build_trucks),
    ("sales",                 build_sales),
    ("sale_lines",            build_sale_lines),
    ("online_orders",         build_online_orders),
    ("online_order_lines",    build_online_order_lines),
    ("payments",              build_payments),
    ("inventory_transactions", build_inventory_transactions),
    ("reorders",              build_reorders),
    ("stockouts",             build_stockouts),
    ("shipment_movements",    build_shipment_movements),
    ("shipment_lines",        build_shipment_lines),
]

if ONLY_TABLES:
    LOAD_PLAN = [(t, fn) for t, fn in LOAD_PLAN if t in ONLY_TABLES]

PLAN_TABLES = [t for t, _ in LOAD_PLAN]

In [ ]:
# =============================================================================
# WRITER
# =============================================================================
import math
from pyspark.storagelevel import StorageLevel


def _stamp_loaded_at(df):
    """Add the single per-run batch timestamp as loaded_at (string -> datetime2
    via implicit conversion, so it is independent of the Spark session tz)."""
    if "loaded_at" in df.columns:
        return df
    return df.withColumn("loaded_at", F.lit(LOADED_AT))


def _write_once(df, table):
    """Single append. Re-acquires auth (fresh AAD token) on every call."""
    df = _stamp_loaded_at(df)
    dbtable = f"{TARGET_SCHEMA}.{table}"
    options = {"url": JDBC_URL, "dbtable": dbtable, "batchsize": str(BATCH_SIZE)}
    options.update(get_auth_options())  # fresh token per physical write
    if WRITE_FORMAT == "jdbc":
        (
            df.write.format("jdbc").mode("append")
            .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
            .options(**options)
            .save()
        )
    else:
        # MSSQL Spark connector (BULK COPY)
        options["tableLock"] = "true"
        options["schemaCheckEnabled"] = "false"
        df.write.format(WRITE_FORMAT).mode("append").options(**options).save()


def _write_table(df, table, row_count=None):
    """Write a DataFrame, chunking large loads so no single connection outlives
    the AAD token. Chunks are deterministic (hash of the row), so a re-scan of
    the source never duplicates or drops rows."""
    if WRITE_PARTITIONS:
        df = df.repartition(WRITE_PARTITIONS)

    chunk = (
        AUTH_MODE == "aad_token"
        and MAX_ROWS_PER_WRITE > 0
        and (row_count or df.count()) > MAX_ROWS_PER_WRITE
    )
    if not chunk:
        _write_once(df, table)
        return

    n = row_count if row_count is not None else df.count()
    chunks = math.ceil(n / MAX_ROWS_PER_WRITE)
    # Persist the hashed frame once so each chunk filter reads from cache instead
    # of recomputing the (often expensive) source join for every chunk.
    keyed = df.withColumn(
        "_chunk", F.pmod(F.hash(*[F.col(c) for c in df.columns]), F.lit(chunks))
    ).persist(StorageLevel.DISK_ONLY)
    try:
        keyed.count()  # materialize the source transform exactly once
        for i in range(chunks):
            part = keyed.filter(F.col("_chunk") == i).drop("_chunk")
            if WRITE_PARTITIONS:
                part = part.repartition(WRITE_PARTITIONS)
            print(f"     chunk {i + 1}/{chunks} (fresh token)")
            _write_once(part, table)  # fresh token acquired inside
    finally:
        keyed.unpersist()


def write_oltp(df, table, row_count=None, attempts=4, base_delay=15):
    """Load one table with retry-with-backoff on transient connection drops.
    On failure the target is TRUNCATEd before retrying. This is duplicate-safe:
    tables load in FK-safe (parent -> child) order and FK enforcement is off, so
    when `table` is being written nothing references its rows yet."""
    for attempt in range(1, attempts + 1):
        try:
            _write_table(df, table, row_count)
            return
        except Exception as exc:  # noqa: BLE001 - Spark/JDBC errors are opaque
            if attempt == attempts:
                raise
            delay = base_delay * (2 ** (attempt - 1))
            print(f"   {table}: write failed ({attempt}/{attempts}): {exc}")
            print(f"   truncating {table} + retrying in {delay}s (no children loaded yet)")
            time.sleep(delay)
            execute_sql([f"TRUNCATE TABLE {TARGET_SCHEMA}.{table}"])


def _plan_list_sql():
    return ", ".join("'" + t.replace("'", "''") + "'" for t in PLAN_TABLES)


def reset_targets():
    """Drop the FK constraints touching the target tables (saving their
    definitions to a backup table) and TRUNCATE each target. TRUNCATE is
    minimally logged, so a reset stays fast even for the 100M+ row fact tables;
    a plain DELETE is I/O-bound and can run for hours clearing leftover rows."""
    plan = _plan_list_sql()
    backup = f"{TARGET_SCHEMA}._fk_backup"
    fk_from = (
        "FROM sys.foreign_keys fk "
        "JOIN sys.tables pt ON pt.object_id = fk.parent_object_id "
        "JOIN sys.schemas ps ON ps.schema_id = pt.schema_id "
        "JOIN sys.tables rt ON rt.object_id = fk.referenced_object_id "
        "JOIN sys.schemas rs ON rs.schema_id = rt.schema_id "
        f"WHERE ps.name = '{TARGET_SCHEMA}' "
        f"AND (pt.name IN ({plan}) OR rt.name IN ({plan}))"
    )
    capture_and_drop = f"""
IF OBJECT_ID('{backup}') IS NULL
    CREATE TABLE {backup} (id INT IDENTITY(1,1) PRIMARY KEY, recreate_sql NVARCHAR(MAX));
DELETE FROM {backup};
INSERT INTO {backup} (recreate_sql)
SELECT 'ALTER TABLE ' + QUOTENAME(ps.name) + '.' + QUOTENAME(pt.name)
     + ' WITH NOCHECK ADD CONSTRAINT ' + QUOTENAME(fk.name) + ' FOREIGN KEY ('
     + STUFF((SELECT ',' + QUOTENAME(pc.name)
              FROM sys.foreign_key_columns k
              JOIN sys.columns pc ON pc.object_id = k.parent_object_id AND pc.column_id = k.parent_column_id
              WHERE k.constraint_object_id = fk.object_id
              ORDER BY k.constraint_column_id FOR XML PATH('')), 1, 1, '')
     + ') REFERENCES ' + QUOTENAME(rs.name) + '.' + QUOTENAME(rt.name) + ' ('
     + STUFF((SELECT ',' + QUOTENAME(rc.name)
              FROM sys.foreign_key_columns k
              JOIN sys.columns rc ON rc.object_id = k.referenced_object_id AND rc.column_id = k.referenced_column_id
              WHERE k.constraint_object_id = fk.object_id
              ORDER BY k.constraint_column_id FOR XML PATH('')), 1, 1, '') + ');'
{fk_from};
DECLARE @drop NVARCHAR(MAX) = N'';
SELECT @drop = @drop + 'ALTER TABLE ' + QUOTENAME(ps.name) + '.' + QUOTENAME(pt.name)
             + ' DROP CONSTRAINT ' + QUOTENAME(fk.name) + ';'
{fk_from};
IF @drop <> N'' EXEC sys.sp_executesql @drop;
"""
    stmts = [capture_and_drop]
    stmts += [f"TRUNCATE TABLE {TARGET_SCHEMA}.{t}" for t in reversed(PLAN_TABLES)]
    execute_sql(stmts)


def reenable_fks():
    """Recreate the FK constraints captured by reset_targets. They are added
    WITH NOCHECK, so there is no validation scan and they stay untrusted
    (matching the prior reset-and-reload behaviour)."""
    backup = f"{TARGET_SCHEMA}._fk_backup"
    recreate = f"""
DECLARE @sql NVARCHAR(MAX) = N'';
SELECT @sql = @sql + recreate_sql FROM {backup} ORDER BY id;
IF @sql <> N'' EXEC sys.sp_executesql @sql;
"""
    execute_sql([recreate])


In [ ]:
# =============================================================================
# RUN
# =============================================================================

if TRUNCATE_BEFORE_LOAD:
    print("Resetting target tables (drop FKs + truncate)...")
    reset_targets()

for table, build in LOAD_PLAN:
    df = build()
    n = df.count()
    print(f"-> {TARGET_SCHEMA}.{table}: writing {n:,} rows")
    write_oltp(df, table, row_count=n)
    print(f"   done: {table}")

if TRUNCATE_BEFORE_LOAD:
    print("Re-enabling FK constraints...")
    reenable_fks()

print("ETL complete.")
